# Test Hybrid Retrieval

## Environment Setup

In [ ]:
# --- JUPYTER MAGIC ---
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
from pathlib import Path

# Add project root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.config import load_settings, root_path
from qdrant_client import QdrantClient
from src.kg.fuseki import FusekiClient

settings = load_settings()
test_db = root_path("data/test_ledger.db")
mock_nextcloud = root_path("notebooks/mock_nextcloud")

# Verify Infrastructure
print("--- INFRASTRUCTURE HEALTH CHECK ---")
fuseki = FusekiClient(settings["kg"]["fuseki_url"], settings["kg"]["dataset"])
print(f"Apache Jena Fuseki: {'✅ ONLINE' if fuseki.health() else '❌ OFFLINE'}")

try:
    QdrantClient(url="http://localhost:6333").get_collections()
    print("Qdrant Vector DB: ✅ ONLINE")
except Exception as e:
    print(f"Qdrant Vector DB: ❌ OFFLINE ({e})")

## Phase 4 - Populate the Brain (Graph + Vectors)

In [ ]:
from src.kg.builder import build_rdf_graph
from src.vector.multimodal import index_unprocessed_files

print("--- 1. BUILDING DETERMINISTIC KNOWLEDGE GRAPH ---")
test_ttl = root_path("artifacts/test_graph.ttl")
build_rdf_graph(test_db, test_ttl)
fuseki.replace_default_graph(test_ttl)
print("✅ Test Graph published to Apache Jena.")

print("\n--- 2. BUILDING PROBABILISTIC VECTOR SPACE ---")
# Note: The first time this runs, it will download the FastEmbed and CLIP models locally!
index_unprocessed_files(
    db_path=test_db,
    nextcloud_mount=mock_nextcloud,
    qdrant_url="http://localhost:6333"
)

# Verify Qdrant
q_client = QdrantClient(url="http://localhost:6333")
collection_info = q_client.get_collection("nas_multimodal")
print(f"\n✅ Qdrant Collection Status: {collection_info.status}")
print(f"Total Vectors Indexed: {collection_info.points_count}")

## Phase 5 - The Hybrid Search Engine

In [ ]:
from src.retrieval.hybrid_search import HybridRetriever
from prefect.blocks.system import Secret
from openai import OpenAI

print("Authenticating with Prefect Vault...")
# Async fetch for Jupyter
llm_key = await Secret.load("nas-gemini-api-key")
llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

retriever = HybridRetriever(
    fuseki_url=settings["kg"]["fuseki_url"],
    qdrant_url="http://localhost:6333",
    llm_client=llm_client,
    model_name=settings["llm"]["model"]
)

print("\n" + "="*50)
print("🔍 TEST 1: Graph-First Query (Deterministic)")
print("="*50)
# The word "pdf" exists in the RDF graph as an extension property. 
# The Graph should intercept this and answer deterministically.
res_graph = retriever.ask("What pdf files do I have?", target_modality="text")
print(f"Source: {res_graph['source']}")
print(f"Answer:\n{res_graph['answer']}\n")

print("="*50)
print("👁️ TEST 2: Multimodal Vector Fallback (Probabilistic)")
print("="*50)
# "Beach" is not a folder name or extension, so the Graph will fail.
# It will automatically fall back to Qdrant, use CLIP to embed the word "beach",
# and find the family_photo.jpg we mocked in Notebook 1!
res_vector = retriever.ask("Find a picture of a beach", target_modality="image")
print(f"Source: {res_vector['source']}")
print(f"Answer:\n{res_vector['answer']}")